# WallGuard 完整版：牆面危害 AI 辨識

拍一張牆面照片，在照片裡分別標出**裂縫、壁癌、鏽蝕**的範圍，換算各自的面積，再產生白話的處理建議。同一張照片有好幾種牆損時，也會分開標示、分開計算。

```
照片 ─▶ U-Net 逐像素分類：背景／裂縫／壁癌／鏽蝕 ─▶ 各自換算面積 ─▶ GPT 處理建議
```

專案開發時試過兩種做法：

- **初版**（`1_resnet_classification.ipynb`）：ResNet50V2 替整張照片分類，知道是哪一種牆損，但看不出位置與大小，一張照片也只能有一個答案。
- **進階版**（`2_unet_segmentation.ipynb`）：U-Net 替每個像素分類，分出各種牆損的範圍並換算面積，再串接 GPT 給建議，也就是展出時的做法。

完整版照展出時的流程重新整理，並改進了實驗階段的問題：

| 實驗階段的問題 | 完整版的做法 |
|---|---|
| 沒有「背景」類別，每個像素都會被歸成某一種牆損，乾淨的牆面也算得出面積 | 分成背景＋三種牆損共四類，只計算牆損像素的面積 |
| 遮罩用固定門檻只分明暗，沒有用到照片的類別：暗的像素被當成鏽蝕、亮的被當成壁癌，裂縫幾乎沒有被標到 | 依類別找出比周圍亮（壁癌）或暗（裂縫、鏽蝕）的區域，標成這張照片的種類；裂縫只保留夠長的線條 |
| 遮罩存成 JPEG，壓縮雜訊變成假的類別 | 標註只放在記憶體，不經過 JPEG |
| 輸入灰階照片，鏽蝕的橘褐色和壁癌的白色差異被抹掉 | 改用彩色照片 |
| 一張訓練照片只有一種牆損，模型沒看過混在一起的情況 | 另外把兩種牆損的照片各取半邊拼成一張，一起訓練 |
| 訓練／驗證集沒有打散，驗證集只有壁癌 | 打散後分層切分，三類都在驗證集 |
| 多類別 softmax 卻用 binary cross-entropy | 改用 cross-entropy＋Dice，並計算三種牆損各自的 IoU |
| 上傳的 RGB 照片用 BGR 轉灰階 | 一律用 RGB 轉換 |

> **限制：** 標註是用影像處理自動產生的近似標註，不是人工標註；混在一起的牆損只用拼接照片模擬；面積以「每個像素 0.5 mm」粗估，沒有依拍攝距離校正。

### 執行前
1. **執行階段 → 變更執行階段類型 → T4 GPU**。
2. 想要 AI 建議的話，在左側欄 🔑 **Secrets** 新增 `OPENAI_API_KEY` 並開啟筆記本存取權；沒有設定也能跑，只是略過建議。
3. 從第一格依序執行。完整訓練在 T4 上約 35～40 分鐘；想先確認流程，把 `QUICK_TEST` 設成 `True`。

## 0. 設定

In [ ]:
QUICK_TEST = False   # True：每類只取 20 張、只訓練 1 個 epoch，只用來確認流程能跑通

CATEGORIES = ["crack", "efflorescence", "concrete"]                       # 資料夾名稱
LABELS_ZH = {"crack": "裂縫", "efflorescence": "壁癌", "concrete": "鏽蝕"}  # 顯示用的名稱
CLASS_NAMES = ["background"] + CATEGORIES   # U-Net 把每個像素分成 4 類：0 是背景，1～3 是三種牆損
NUM_CLASSES = len(CLASS_NAMES)
COLORS = {"crack": (230, 40, 40), "efflorescence": (30, 110, 255), "concrete": (255, 190, 0)}  # 疊圖顏色
COLOR_NAMES = {"crack": "紅", "efflorescence": "藍", "concrete": "黃"}
# 產生近似標註的規則：缺損比周圍亮（壁癌）還是暗（裂縫、鏽蝕）、比較的範圍（像素）、
# 只取反差最強的前幾 %、最短長度（裂縫只保留夠長的線條）
MASK_RULES = {
    "crack":         dict(bright=False, kernel=21, percentile=93, min_length=30),
    "efflorescence": dict(bright=True,  kernel=31, percentile=85, min_length=0),
    "concrete":      dict(bright=False, kernel=25, percentile=90, min_length=0),
}

DATASET_URL = "https://github.com/Yunyuhu/dtaidataset/raw/main/dataset.zip"
IMG_SIZE = 224
VAL_RATIO = 0.2
SEED = 42
MIX_RATIO = 0.3     # 另外拼出訓練集 30% 數量的混合照片：左右半邊各是一種牆損

UNET_EPOCHS, UNET_BATCH = 10, 4        # batch size 沿用展出時的 4

PIXEL_MM = 0.5          # 和展出時一樣，假設每個像素 0.5 mm × 0.5 mm（沒有依拍攝距離校正）
MIN_COVERAGE = 0.002    # 某種牆損占照片不到 0.2%，當作雜訊不列出
OPENAI_MODEL = "gpt-4o-mini"   # 請依 OpenAI 目前提供的模型調整

if QUICK_TEST:
    UNET_EPOCHS = 1

## 1. 安裝與匯入套件

In [ ]:
!pip install -q gradio openai

In [ ]:
import os, random, zipfile, urllib.request
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import gradio as gr
from openai import OpenAI

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
WORK_DIR = "/content" if os.path.isdir("/content") else os.path.abspath("wallguard_work")
os.makedirs(WORK_DIR, exist_ok=True)
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "沒有偵測到 GPU，訓練會非常慢")

In [ ]:
# OpenAI 金鑰：優先讀 Colab Secrets，其次讀環境變數；都沒有就略過 AI 建議
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key) if api_key else None
print("AI 建議：", "已啟用" if client else "未設定 OPENAI_API_KEY，將略過")

## 2. 下載資料並切分

3,000 張室內牆面照片，三類各 1,000 張，來自 Roboflow Universe 的公開資料集。依種類分層切出 20% 當驗證集，訓練時不會看到。

In [ ]:
zip_path = os.path.join(WORK_DIR, "dataset.zip")
data_dir = os.path.join(WORK_DIR, "dataset")
if not os.path.exists(zip_path):
    print("下載資料集（約 190 MB）…")
    urllib.request.urlretrieve(DATASET_URL, zip_path)
if not os.path.isdir(data_dir):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(data_dir)

paths, labels = [], []
for k, cat in enumerate(CATEGORIES):
    folder = os.path.join(data_dir, cat)
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"資料集裡找不到 {cat}/ 資料夾，請確認 DATASET_URL 是否正確")
    names = sorted(f for f in os.listdir(folder)
                   if f.lower().endswith((".jpg", ".jpeg")) and not f.startswith("._"))
    if QUICK_TEST:
        names = names[:20]
    paths += [os.path.join(folder, f) for f in names]
    labels += [k] * len(names)
paths, labels = np.array(paths), np.array(labels)

train_p, val_p, train_y, val_y = train_test_split(
    paths, labels, test_size=VAL_RATIO, stratify=labels, random_state=SEED)
for name, y in (("訓練", train_y), ("驗證", val_y)):
    print(name, len(y), "張：", {LABELS_ZH[c]: int((y == k).sum()) for k, c in enumerate(CATEGORIES)})

In [ ]:
def read_rgb(path, size=IMG_SIZE):
    """讀成 RGB、縮放成 size×size 的 uint8 陣列。"""
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"讀不到圖片：{path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

train_rgb = np.stack([read_rgb(p) for p in train_p])
val_rgb = np.stack([read_rgb(p) for p in val_p])
print("訓練", train_rgb.shape, "驗證", val_rgb.shape)

fig, axes = plt.subplots(1, len(CATEGORIES), figsize=(9, 3))
for ax, (k, cat) in zip(axes, enumerate(CATEGORIES)):
    ax.imshow(train_rgb[np.where(train_y == k)[0][0]]); ax.set_title(cat); ax.axis("off")
plt.show()

## 3. 產生標註：每個像素屬於哪一種牆損

資料集只標了「整張照片是哪一種牆損」，沒有逐像素的標註，所以先用影像處理自動產生：

- 用形態學的 **top-hat** 找出比周圍亮的區域（壁癌），**black-hat** 找出比周圍暗的區域（裂縫、鏽蝕）。只取反差最強的一部分，再去掉零碎的小點；裂縫只保留夠長的線條。
- 找到的像素標成這張照片的種類（1＝裂縫、2＝壁癌、3＝鏽蝕），其他像素是背景 0。
- 每張訓練照片只有一種牆損，所以另外把兩張不同種類的照片各取半邊拼成一張，讓模型看過「同一張照片有兩種牆損」的情況。

> 這仍是自動產生的近似標註。例如裂縫資料夾裡混有沒有裂縫的牆面照片，標註會標到一些紋理。

In [ ]:
def to_gray(rgb):
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

def make_mask(rgb, category):
    """依類別找出缺損的像素：缺損＝1、背景＝0。"""
    rule = MASK_RULES[category]
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (rule["kernel"], rule["kernel"]))
    op = cv2.MORPH_TOPHAT if rule["bright"] else cv2.MORPH_BLACKHAT
    response = cv2.GaussianBlur(cv2.morphologyEx(to_gray(rgb), op, kernel), (3, 3), 0)
    mask = (response >= max(np.percentile(response, rule["percentile"]), 12)).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))   # 接上斷掉的線段
    _, comp, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    longest = np.maximum(stats[:, cv2.CC_STAT_WIDTH], stats[:, cv2.CC_STAT_HEIGHT])
    keep = (stats[:, cv2.CC_STAT_AREA] >= 40) & (longest >= rule["min_length"])
    keep[0] = False                                                              # 0 是背景
    return keep[comp].astype(np.uint8)

def make_label(rgb, k):
    """逐像素標註：缺損像素標成這張照片的種類編號（1～3），其他像素是背景 0。"""
    return make_mask(rgb, CATEGORIES[k]) * np.uint8(k + 1)

def make_mosaics(rgb, label, y, n, seed=SEED):
    """把兩張不同種類照片的左右半邊拼成一張；回傳拼接照片、標註，以及左右兩邊的種類。"""
    rng = np.random.default_rng(seed)
    half = IMG_SIZE // 2
    a = rng.integers(0, len(y), n)
    b = np.array([rng.choice(np.where(y != y[i])[0]) for i in a], dtype=int)
    mix_rgb = np.concatenate([rgb[a, :, :half], rgb[b, :, half:]], axis=2)
    mix_label = np.concatenate([label[a, :, :half], label[b, :, half:]], axis=2)
    return mix_rgb, mix_label, np.stack([y[a], y[b]], axis=1)

PALETTE = np.array([(0, 0, 0)] + [COLORS[c] for c in CATEGORIES], dtype=np.float32)   # 依類別編號查顏色

def overlay(rgb, label, alpha=0.5):
    """把各種牆損用各自的顏色疊在照片上；label 比照片小時會先放大。"""
    if label.shape != rgb.shape[:2]:
        label = cv2.resize(label, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
    out = rgb.copy()
    hit = label > 0
    out[hit] = ((1 - alpha) * out[hit] + alpha * PALETTE[label[hit]]).astype(np.uint8)
    return out

train_label = np.stack([make_label(x, k) for x, k in zip(train_rgb, train_y)])
val_label = np.stack([make_label(x, k) for x, k in zip(val_rgb, val_y)])
mix_train_rgb, mix_train_label, _ = make_mosaics(train_rgb, train_label, train_y, int(len(train_y) * MIX_RATIO))
mix_val_rgb, mix_val_label, mix_val_pairs = make_mosaics(val_rgb, val_label, val_y, len(val_y) // 2, seed=SEED + 1)
print("標註中牆損像素的比例（訓練集）：", {LABELS_ZH[c]: f"{(train_label[train_y == k] > 0).mean():.1%}"
                                for k, c in enumerate(CATEGORIES)})
print(f"拼接照片：訓練 {len(mix_train_rgb)} 張、驗證 {len(mix_val_rgb)} 張")
print("疊圖顏色：", "、".join(f"{COLOR_NAMES[c]}＝{LABELS_ZH[c]}" for c in CATEGORIES))

cols = [(train_rgb[i], train_label[i], CATEGORIES[train_y[i]])
        for i in (int(np.where(train_y == k)[0][0]) for k in range(len(CATEGORIES)))]
cols.append((mix_train_rgb[0], mix_train_label[0], "mosaic"))
fig, axes = plt.subplots(2, len(cols), figsize=(3 * len(cols), 6))
for j, (img, lab, title) in enumerate(cols):       # Colab 沒有中文字型，圖上用英文
    axes[0, j].imshow(img); axes[0, j].set_title(title)
    axes[1, j].imshow(overlay(img, lab)); axes[1, j].set_title("label")
for ax in axes.ravel(): ax.axis("off")
plt.show()

## 4. 訓練 U-Net：逐像素分成背景與三種牆損

U-Net 架構沿用展出時的版本（4 次卷積與池化擷取特徵，再上採樣、跳接還原尺寸），輸入改成彩色照片，最後一層輸出 4 類的 softmax。牆損通常只占照片一小部分，所以損失函數用 cross-entropy 加上三種牆損的 Dice，避免模型全部猜背景。

In [ ]:
def make_seg_ds(rgb, label, training):
    ds = tf.data.Dataset.from_tensor_slices((rgb, label[..., None]))   # 先以 uint8 存放，節省記憶體
    if training:
        ds = ds.shuffle(len(rgb), seed=SEED)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, tf.cast(y, tf.float32)),
                num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        def augment(img, lab):
            both = tf.concat([img, lab], axis=-1)          # 照片與標註做同樣的翻轉與旋轉
            both = tf.image.random_flip_left_right(both)
            both = tf.image.random_flip_up_down(both)
            both = tf.image.rot90(both, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
            return both[..., :3], both[..., 3:]
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(UNET_BATCH).prefetch(tf.data.AUTOTUNE)

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    return layers.Conv2D(filters, 3, activation="relu", padding="same")(x)

def unet(input_size=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    inputs = keras.Input(input_size)
    skips, x = [], inputs
    for f in (64, 128, 256, 512):                         # 編碼：擷取特徵
        x = conv_block(x, f); skips.append(x)
        x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, 1024)
    for f, skip in zip((512, 256, 128, 64), reversed(skips)):   # 解碼：上採樣並拼接還原尺寸
        x = layers.UpSampling2D(2)(x)
        x = conv_block(layers.concatenate([skip, x]), f)
    outputs = layers.Conv2D(num_classes, 1, activation="softmax")(x)   # 每個像素 4 類的機率
    return keras.Model(inputs, outputs, name="unet")

def ce_dice_loss(y_true, y_pred):
    """cross-entropy＋三種牆損的平均 Dice：背景占大部分時，Dice 讓模型不會全部猜背景。"""
    y_true = tf.cast(y_true[..., 0], tf.int32)
    ce = tf.reduce_mean(keras.losses.sparse_categorical_crossentropy(y_true, y_pred))
    truth = tf.one_hot(y_true, NUM_CLASSES)[..., 1:]          # 只算三種牆損，不算背景
    pred = y_pred[..., 1:]
    inter = tf.reduce_sum(truth * pred, axis=[0, 1, 2])
    total = tf.reduce_sum(truth + pred, axis=[0, 1, 2])
    return ce + tf.reduce_mean(1.0 - (2.0 * inter + 1.0) / (total + 1.0))

train_x, train_t = np.concatenate([train_rgb, mix_train_rgb]), np.concatenate([train_label, mix_train_label])
val_x, val_t = np.concatenate([val_rgb, mix_val_rgb]), np.concatenate([val_label, mix_val_label])

seg_model = unet()
seg_model.compile(optimizer=keras.optimizers.Adam(1e-4), loss=ce_dice_loss,
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")])
print(f"U-Net 參數量：{seg_model.count_params():,}；訓練 {len(train_x)} 張（含拼接 {len(mix_train_rgb)} 張）")

seg_history = seg_model.fit(
    make_seg_ds(train_x, train_t, True),
    validation_data=make_seg_ds(val_x, val_t, False),
    epochs=UNET_EPOCHS,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=2, min_lr=1e-7)])

In [ ]:
def predict_labels(rgb, batch=16):
    """逐批預測，回傳每個像素機率最高的類別編號（0 背景、1～3 牆損）。"""
    out = []
    for i in range(0, len(rgb), batch):
        prob = seg_model.predict_on_batch(rgb[i:i + batch].astype("float32") / 255.0)
        out.append(np.asarray(prob).argmax(axis=-1).astype(np.uint8))
    return np.concatenate(out)

def iou_by_class(pred, truth):
    """三種牆損各自的 IoU：預測和標註重疊的像素 ÷ 兩者合起來的像素。"""
    result = {}
    for c in range(1, NUM_CLASSES):
        union = np.logical_or(pred == c, truth == c).sum()
        result[LABELS_ZH[CLASS_NAMES[c]]] = np.logical_and(pred == c, truth == c).sum() / union if union else np.nan
    return result

val_pred, mix_pred = predict_labels(val_rgb), predict_labels(mix_val_rgb)
for name, pred, truth in (("單一種類的驗證照片", val_pred, val_label), ("拼接的混合照片", mix_pred, mix_val_label)):
    ious = iou_by_class(pred, truth)
    print(f"{name}：像素準確率 {(pred == truth).mean():.2%}｜IoU",
          "、".join(f"{k} {v:.3f}" for k, v in ious.items()), f"｜平均 {np.nanmean(list(ious.values())):.3f}")

# 以照片為單位：面積最大的牆損種類，是不是照片所屬的資料夾
counts = np.stack([(val_pred == c).sum(axis=(1, 2)) for c in range(1, NUM_CLASSES)], axis=1)
main = np.where(counts.sum(axis=1) > 0, counts.argmax(axis=1), -1)
print(f"以照片為單位，面積最大的種類＝資料夾種類：{(main == val_y).mean():.2%}"
      f"（{int((main == -1).sum())} 張沒有偵測到牆損）")
print("（比對的是自動產生的近似標註，不是人工標註）")

rows = [(val_rgb[i], val_label[i], val_pred[i], CATEGORIES[val_y[i]])
        for i in (int(np.where(val_y == k)[0][0]) for k in range(len(CATEGORIES)))]
rows.append((mix_val_rgb[0], mix_val_label[0], mix_pred[0], "mosaic"))
fig, axes = plt.subplots(len(rows), 3, figsize=(8, 2.8 * len(rows)))
for row, (img, truth, pred, title) in zip(axes, rows):
    for ax, im, t in zip(row, (img, overlay(img, truth), overlay(img, pred)),
                         (title, "pseudo label", "U-Net prediction")):   # Colab 沒有中文字型，圖上用英文
        ax.imshow(im); ax.set_title(t); ax.axis("off")
plt.show()

## 5. 整合：範圍＋面積＋建議

照片經過 U-Net 後，每個像素都有一個類別。三種牆損分別換算占照片的比例與面積，再把結果交給 GPT 產生白話的處理建議。

In [ ]:
SYSTEM_PROMPT = (
    "你是協助一般住戶理解牆面問題的助理。照片裡可能同時有好幾種牆損，請用繁體中文、白話、條列回答：\n"
    "1. 每一種問題可能的成因\n"
    "2. 嚴重程度，以及是否需要找專業人員\n"
    "3. 住戶可以先做的處理；有好幾種問題時，說明先處理哪一個\n"
    "最後提醒：這只是 AI 的初步判斷，涉及結構安全時應請專業技師到場評估。")

def get_advice(findings):
    """findings：[(種類, 占照片比例, 面積 mm²), …]，已依面積由大到小排好。"""
    if not findings:
        return "沒有偵測到明顯的牆損。若牆面仍有疑慮，建議請專業人員到場查看。"
    if client is None:
        return "（未設定 OPENAI_API_KEY，略過 AI 建議）"
    summary = ("照片中偵測到：" +
               "；".join(f"{LABELS_ZH[c]}約占照片 {cov:.1%}，粗估 {area:,.0f} mm²" for c, cov, area in findings) +
               f"。面積以每像素 {PIXEL_MM} mm 估算，沒有依拍攝距離校正。")
    try:
        response = client.chat.completions.create(
            model=OPENAI_MODEL, temperature=0.5, max_tokens=500,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": summary}])
        return response.choices[0].message.content
    except Exception as e:
        return f"AI 建議產生失敗：{e}"

def analyze(image):
    """輸入 RGB 照片，回傳各種牆損的疊圖、面積說明與處理建議。"""
    if image is None:
        raise gr.Error("請先上傳一張牆面照片")
    image = np.asarray(image)[..., :3].astype(np.uint8)
    rgb = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    # 1) 逐像素分類：每個像素是背景、裂縫、壁癌或鏽蝕
    prob = seg_model.predict(rgb[None].astype("float32") / 255.0, verbose=0)[0]
    label = prob.argmax(axis=-1).astype(np.uint8)

    # 2) 三種牆損分別換算面積；占比太小的當作雜訊
    findings = []
    for c in range(1, NUM_CLASSES):
        pixels = int((label == c).sum())
        if pixels / label.size >= MIN_COVERAGE:
            findings.append((CLASS_NAMES[c], pixels / label.size, pixels * PIXEL_MM ** 2))
        else:
            label[label == c] = 0                  # 不列出的也不畫
    findings.sort(key=lambda f: f[2], reverse=True)

    # 3) 面積說明與建議
    if findings:
        area_text = "\n".join(f"{LABELS_ZH[c]}（{COLOR_NAMES[c]}色）：約占照片 {cov:.1%}，粗估 {area:,.0f} mm²"
                              for c, cov, area in findings)
        area_text += f"\n（假設每像素 {PIXEL_MM} mm，沒有依拍攝距離校正）"
    else:
        area_text = "沒有偵測到明顯的牆損"
    return overlay(image, label), area_text, get_advice(findings)

In [ ]:
# 用驗證集每類一張照片，再加一張拼接的混合照片，試跑整條流程
tests = [(f"資料夾：{LABELS_ZH[c]}", read_rgb(val_p[int(np.where(val_y == k)[0][0])], size=448))
         for k, c in enumerate(CATEGORIES)]
left, right = (LABELS_ZH[CATEGORIES[k]] for k in mix_val_pairs[0])
tests.append((f"拼接：左半{left}、右半{right}", mix_val_rgb[0]))
for title, img in tests:
    result, area_text, advice = analyze(img)
    print(title); print(area_text)
    plt.figure(figsize=(3, 3)); plt.imshow(result); plt.axis("off"); plt.show()
print("建議範例（最後一張）：\n", advice)

## 6. 互動介面

上傳牆面照片，一次看到各種牆損的範圍、面積與建議。範例最後一張是拼接的混合照片。

In [ ]:
mix_path = os.path.join(WORK_DIR, "example_mixed.jpg")
cv2.imwrite(mix_path, cv2.cvtColor(mix_val_rgb[0], cv2.COLOR_RGB2BGR))
examples = [val_p[int(np.where(val_y == k)[0][0])] for k in range(len(CATEGORIES))] + [mix_path]
legend = "、".join(f"{COLOR_NAMES[c]}＝{LABELS_ZH[c]}" for c in CATEGORIES)
demo = gr.Interface(
    fn=analyze,
    inputs=gr.Image(type="numpy", label="牆面照片"),
    outputs=[gr.Image(label=f"牆損範圍（{legend}）"),
             gr.Textbox(label="各種牆損的面積"),
             gr.Textbox(label="AI 處理建議", lines=10)],
    title="WallGuard 牆面危害辨識（完整版）",
    description="上傳一張室內牆面照片，分別標出裂縫、壁癌與鏽蝕的範圍，估計各自的面積，再給出處理建議。結果僅供初步參考。",
    examples=examples)
demo.launch(debug=True)

## 7. 儲存模型（可選）

模型檔很大（U-Net 約 120 MB），沒有放進 GitHub。需要保留時，執行下面這格後從左側「檔案」下載，或複製到自己的 Google 雲端硬碟。

In [ ]:
model_dir = os.path.join(WORK_DIR, "models")
os.makedirs(model_dir, exist_ok=True)
seg_model.save(os.path.join(model_dir, "wallguard_unet.keras"))
print("已儲存到", model_dir)
# 之後要載入：keras.models.load_model(路徑, compile=False)